In [1]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
granularity = 'q'

# --- Date Range (inclusive) ---
START_DATE = '2020-01-01'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---
run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

# --- Rollup Groups (weighted-average aggregation of individual LOB results) ---
ROLLUP_GROUPS = {
    'Franchise Independent': ['AN', 'FLD', 'FRN', 'STG'],
    'nonKMX': ['AN', 'FRN', 'STG', 'FLD', 'ENT'],
    'POS': ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX'],
}

# --- Single Baseline for All LOBs ---
BASELINE = {'ltv': 1.5, 'apr': 0.24}

# --- Model Parameters (V2) ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'find_rate': 0.75,
    'unit_loss_to_model_score': 0.027,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
    'kmx_bps_per_point': 0.65,
    'kmx_ms_offset': 50,
}

# --- DLA (Dealer Level Adjustment) Current Quarter ---
DLA_CURRENT_QUARTER = '2026 Q1'

# --- Excluded Vintages (per LOB) ---
EXCLUDED_VINTAGES = {}

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: q
Date column: book_date
Period range: 2020Q1 to 2026Q3
SQL min_date: '2020-01-01'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS + V2 FORMULA DEFINITIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    """Per-metric population-aware weighted average: each metric's denominator
    only includes amt_financed from accounts where that metric is not NaN."""
    if isinstance(metrics, str):
        valid = group[metrics].notna()
        if valid.any():
            weighted_avg = (group.loc[valid, metrics] * group.loc[valid, 'amt_financed']).sum() / group.loc[valid, 'amt_financed'].sum()
        else:
            weighted_avg = np.nan
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        valid = group[metric].notna()
        if valid.any():
            result_dict[metric] = (
                (group.loc[valid, metric] * group.loc[valid, 'amt_financed']).sum()
                / group.loc[valid, 'amt_financed'].sum()
            )
        else:
            result_dict[metric] = np.nan
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings (e.g. '2025 Q1', '2025 M01')."""
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


# =============================================================================
# RAGU V2 FORMULA DEFINITIONS
# =============================================================================

def compute_unit_loss(model_score, term):
    """Actual unit loss derived from model score and loan term."""
    return 0.9 - (model_score - 125) * 0.027 - (72 - term) / 240

def compute_gross_loss_impact(loss_multiplier, unit_loss, ul_to_ms=0.027):
    """Gross loss impact: deviation from baseline loss, converted to score points.
    UL replaces the old hardcoded 0.5 approximation.
    ul_to_ms updated from 0.02 to 0.027."""
    return (1 - loss_multiplier) * unit_loss / ul_to_ms

def compute_recovery_impact(unit_loss, find_rate, recovery_multiplier, ul_to_ms=0.027):
    """Recovery impact: baseline-free, converted to score points using same
    factor as gross loss for economic consistency ($1 lost = $1 recovered).
    Neutral at R=0 (no recovery). UL weights recovery by default probability;
    find_rate (F) scales by repo success rate."""
    return unit_loss * find_rate * recovery_multiplier / ul_to_ms

def compute_ltv_impact(baseline_ltv, actual_ltv, ltv_mult=17):
    """LTV impact: linear deviation from baseline.
    Replaces the old 1/LTV nonlinear form."""
    return (baseline_ltv - actual_ltv) * ltv_mult

def compute_apr_impact(baseline_apr, actual_apr, apr_mult=0.7):
    """APR impact: linear deviation from baseline. Unchanged."""
    return (baseline_apr - actual_apr) / 0.01 * apr_mult

def rescale_kmx(value, bps_per_point=0.65, ms_offset=50, is_model_score=False):
    """Rescale KMX values from 65bps/pt to 100bps/pt space.
    Model score gets an offset to keep it in the familiar ~140 range."""
    scaled = value * bps_per_point
    if is_model_score:
        scaled += ms_offset
    return scaled

In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_v2.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query_v2.txt', '../../cache/ula_v2.pkl',
            sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
        )
        print('ULA ready (V2 with con_term)')

        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_v2.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")

ULA ready (V2 with con_term)
DLA ready
New recovery ready
ULA records: 2,227,430


In [6]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING + FLAG CREATION
# =============================================================================

# Filter out Core LOB
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns using pd.to_period
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")

# =============================================================================
# FLAG CREATION AND DATA REFINEMENT
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag (derived from ULA job_company) ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- ULA NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# Convert period to string for vintage-based lookups
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- MTN 4.1 model score transformation (applied AFTER flag creation) ---
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"MTN 4.1 accounts translated: {is_mtn41.sum():,}")

Periods in data: 27
Period range: 2020Q1 to 2026Q3
ULA after refinement: 869,379
MTN 4.1 accounts translated: 6,328


In [7]:
# =============================================================================
# CELL 7: ACCOUNT-LEVEL RAGU V2 COMPUTATION
# =============================================================================
# V2 formulas applied per individual loan. All formulas are linear in the
# loan-level inputs, so E[f(x)] = f(E[x]) -- no Jensen correction needed.

unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']
find_rate = MODEL_PARAMS['find_rate']
kmx_bps_per_point = MODEL_PARAMS['kmx_bps_per_point']
kmx_ms_offset = MODEL_PARAMS['kmx_ms_offset']
baseline_ltv = BASELINE['ltv']
baseline_apr = BASELINE['apr']

# --- Apply ULA multipliers (all accounts at once, split by KMX vs nonKMX) ---
kmx_mask = ula_df_total.lob == 'KMX'
ula_kmx = get_ula_multiplier_kmx(ula_df_total[kmx_mask].copy())
ula_nonkmx = get_ula_multiplier_nonkmx(ula_df_total[~kmx_mask].copy())
ula_all = pd.concat([ula_kmx, ula_nonkmx])

# --- Merge recovery multiplier (left join to preserve full population) ---
nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')
acct_df = ula_all.merge(
    nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
    on='account_number', how='left'
).drop_duplicates(subset='account_number', keep='first')

has_bb = acct_df['bbvalue'].notna() & (acct_df['bbvalue'] > 0)

print(f"Total accounts: {len(acct_df):,}")
print(f"  with bbvalue > 0: {has_bb.sum():,}")
print(f"  with recovery: {acct_df['recovery_multiplier'].notna().sum():,}")

# --- Per-account V2 RAGU computation ---
acct_df['ltv'] = np.where(has_bb, acct_df.amt_financed / acct_df.bbvalue, np.nan)
acct_df['model_score'] = acct_df.cd_model_score.copy()

# V2: unit loss derived from model score and loan term
acct_df['unit_loss'] = compute_unit_loss(acct_df.cd_model_score, acct_df.con_term)

# V2: gross loss impact using actual UL and 0.027 conversion
acct_df['gross_loss_impact'] = compute_gross_loss_impact(
    acct_df.loss_multiplier, acct_df.unit_loss, unit_loss_to_model_score)

# V2: baseline-free recovery (UL * F * R / 0.027)
acct_df['recovery_impact'] = compute_recovery_impact(
    acct_df.unit_loss, find_rate, acct_df.recovery_multiplier, unit_loss_to_model_score)

# V2: linear LTV impact with single baseline
acct_df['ltv_impact'] = compute_ltv_impact(baseline_ltv, acct_df.ltv)

# V2: APR impact with single baseline
acct_df['apr_impact'] = compute_apr_impact(baseline_apr, acct_df.apr)

# --- KMX rescaling (uniform * 0.65 on all components, + 50 on model score) ---
kmx = acct_df.lob == 'KMX'
acct_df.loc[kmx, 'model_score'] = rescale_kmx(
    acct_df.loc[kmx, 'model_score'], kmx_bps_per_point, kmx_ms_offset, is_model_score=True)
for col in ['gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact']:
    acct_df.loc[kmx, col] *= kmx_bps_per_point

# --- Final RAGU score ---
acct_df['ragu_score'] = (
    acct_df.model_score
    + acct_df.gross_loss_impact
    + acct_df.recovery_impact
    + acct_df.ltv_impact
    + acct_df.apr_impact
)

# Select output columns
output_cols = [
    'account_number', 'lob', 'vintage', 'book_date', 'app_date', 'model_score',
    'unit_loss', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed',
    'ltv', 'apr', 'recovery_multiplier', 'loss_multiplier', 'con_term',
]
acct_df = acct_df[output_cols].copy()

print(f"Account-level RAGU V2 computed: {len(acct_df):,} accounts")
print(f"  with full RAGU (bbvalue + recovery): {acct_df['ragu_score'].notna().sum():,}")
print(f"LOBs: {sorted(acct_df.lob.unique())}")
print(f"Vintages: {acct_df.vintage.nunique()}")
display(acct_df.head(20))

Total accounts: 863,759
  with bbvalue > 0: 774,039
  with recovery: 841,074
Account-level RAGU V2 computed: 863,759 accounts
  with full RAGU (bbvalue + recovery): 768,749
LOBs: ['', 'AN', 'ENT', 'FLD', 'FRN', 'KMX', 'MCY', 'ORL', 'STE', 'STG', 'unassigned']
Vintages: 27


,account_number,lob,vintage,book_date,app_date,model_score,unit_loss,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,amt_financed,ltv,apr,recovery_multiplier,loss_multiplier,con_term
0,9.012401e+10,KMX,2020 Q4,2020-12-28,2020-12-22,131.90,0.873,2.215531,9.496429,-2.977234,-0.04550,140.589226,15571.01,1.769433,0.2410,0.602470,0.894582,72.0
1,9.012389e+10,KMX,2020 Q2,2020-05-20,2020-05-12,142.95,0.414,1.550023,3.589671,-14.433500,1.54700,135.203193,15504.25,2.806199,0.2060,0.480223,0.844479,72.0
2,9.012397e+10,KMX,2020 Q3,2020-09-28,2020-09-25,133.85,0.792,0.276899,9.370722,-9.615466,-0.04550,133.836655,15583.92,2.370178,0.2410,0.655295,0.985477,72.0
3,9.012402e+10,KMX,2021 Q1,2021-01-11,2021-01-05,144.90,0.333,0.458945,3.387634,1.054530,2.27500,152.076109,18540.29,1.404567,0.1900,0.563432,0.942751,72.0
4,9.012400e+10,KMX,2020 Q4,2020-11-30,2020-11-27,139.05,0.576,1.639137,5.937686,2.168880,1.54700,150.342703,15905.40,1.303721,0.2060,0.570931,0.881793,72.0
5,9.012388e+10,KMX,2020 Q2,2020-04-29,2020-04-25,125.40,1.143,6.536881,11.727118,NaN,-1.81545,NaN,15192.39,NaN,0.2799,0.568243,0.762439,72.0
6,9.012384e+10,KMX,2020 Q1,2020-02-24,2020-02-22,131.90,0.873,4.037905,11.186571,-0.257864,-1.82000,145.046612,20031.87,1.523336,0.2800,0.709695,0.807871,72.0
7,9.012403e+10,KMX,2021 Q1,2021-01-25,2021-01-22,141.00,0.495,2.879955,4.946097,0.761966,-0.04550,149.542519,38566.63,1.431044,0.2410,0.553409,0.758325,72.0
8,9.012384e+10,KMX,2020 Q1,2020-02-21,2020-02-20,137.10,0.657,3.800797,7.247315,NaN,-1.82000,NaN,21748.92,NaN,0.2800,0.610943,0.759697,72.0
9,9.012397e+10,KMX,2020 Q4,2020-10-05,2020-10-01,148.15,0.198,0.542972,3.113784,4.098721,1.36500,157.270477,20238.67,1.129075,0.2100,0.870989,0.886090,72.0


In [8]:
# =============================================================================
# CELL 8: VINTAGE-LOB AGGREGATION + ROLLUPS
# =============================================================================

agg_metrics = [
    'model_score', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
]

# --- Aggregate account-level to vintage-LOB level ---
vintage_lob_df = acct_df.groupby(['vintage', 'lob']).apply(
    weighted_average_and_sum, agg_metrics, include_groups=False
).reset_index()

print(f"Vintage-LOB aggregation: {len(vintage_lob_df)} rows "
      f"({vintage_lob_df.vintage.nunique()} vintages x {vintage_lob_df.lob.nunique()} LOBs)")

# --- Rollup groups ---
for group_name, group_lobs in ROLLUP_GROUPS.items():
    group = vintage_lob_df[vintage_lob_df.lob.isin(group_lobs)].copy()
    rollup = group.groupby('vintage').apply(
        weighted_average_and_sum, agg_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = group_name
    vintage_lob_df = pd.concat([vintage_lob_df, rollup], ignore_index=True)

print(f"After rollups: {len(vintage_lob_df)} rows across {vintage_lob_df.lob.nunique()} groups")
print(f"Groups: {sorted(vintage_lob_df.lob.unique())}")

display(vintage_lob_df.sort_values(['lob', 'vintage']).head(30))

# --- Diagnostic: population breakdown per vintage-LOB ---
print("\n--- Population breakdown (first 5 vintage-LOB combos) ---")
for (vintage, lob), grp in list(acct_df.groupby(['vintage', 'lob']))[:5]:
    n_total = len(grp)
    n_bb = grp['ltv'].notna().sum()
    n_ragu = grp['ragu_score'].notna().sum()
    af_total = grp['amt_financed'].sum()
    af_bb = grp.loc[grp['ltv'].notna(), 'amt_financed'].sum()
    af_ragu = grp.loc[grp['ragu_score'].notna(), 'amt_financed'].sum()
    print(f"  {lob} {vintage}: {n_total:,} total | {n_bb:,} bbvalue>0 | {n_ragu:,} scored | "
          f"AF total={af_total:,.0f} | AF bb={af_bb:,.0f} | AF scored={af_ragu:,.0f}")

Vintage-LOB aggregation: 207 rows (27 vintages x 11 LOBs)
After rollups: 288 rows across 14 groups
Groups: ['', 'AN', 'ENT', 'FLD', 'FRN', 'Franchise Independent', 'KMX', 'MCY', 'ORL', 'POS', 'STE', 'STG', 'nonKMX', 'unassigned']


,vintage,lob,amt_financed,model_score,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,ltv,apr
0,2020 Q1,,111357.46,131.527448,-0.729925,NaN,2.361504,-1.888001,NaN,1.361088,0.266971
9,2020 Q2,,67289.38,138.304012,0.783636,NaN,-3.560122,-1.767442,NaN,1.709419,0.265249
1,2020 Q1,AN,32298104.24,131.797635,1.138435,11.162065,-6.903844,-0.204115,136.846838,1.906108,0.242916
10,2020 Q2,AN,28044942.63,133.710593,0.964018,10.211887,-7.638385,-0.624934,136.606377,1.949317,0.248928
18,2020 Q3,AN,26921133.80,135.285574,0.239732,9.762005,-3.046648,-0.398570,141.777365,1.679215,0.245694
25,2020 Q4,AN,22337712.47,135.796084,0.654051,9.731006,-2.871251,0.065034,143.226316,1.668897,0.239071
32,2021 Q1,AN,32920183.78,134.601788,0.996316,10.213188,-3.288618,-0.546693,141.956642,1.693448,0.247810
39,2021 Q2,AN,39867774.29,135.142573,0.967398,9.940289,-0.353515,-0.420362,145.216754,1.520795,0.246005
46,2021 Q3,AN,26702547.95,135.358873,0.708906,9.786489,-1.548104,-0.567866,143.714297,1.591065,0.248112
53,2021 Q4,AN,23414496.05,134.415379,0.374927,10.012046,-1.476824,-0.542000,142.686114,1.586872,0.247743



--- Population breakdown (first 5 vintage-LOB combos) ---
   2020 Q1: 8 total | 8 bbvalue>0 | 0 scored | AF total=111,357 | AF bb=111,357 | AF scored=0
  AN 2020 Q1: 2,293 total | 2,274 bbvalue>0 | 2,036 scored | AF total=32,298,104 | AF bb=31,963,358 | AF scored=28,592,260
  ENT 2020 Q1: 2,509 total | 2,485 bbvalue>0 | 2,229 scored | AF total=43,073,153 | AF bb=42,677,374 | AF scored=37,947,462
  FLD 2020 Q1: 1,810 total | 1,782 bbvalue>0 | 1,605 scored | AF total=29,625,806 | AF bb=29,160,707 | AF scored=26,126,020
  FRN 2020 Q1: 1,750 total | 1,732 bbvalue>0 | 1,585 scored | AF total=25,578,315 | AF bb=25,347,800 | AF scored=23,175,279


In [9]:
# =============================================================================
# CELL 9: EXCEL EXPORT + REDSHIFT UPLOAD
# =============================================================================

EXCEL_SHEET_MAP = {'q': 'Individual (Q)', 'm': 'Individual (M)', 'w': 'Individual (W)'}
AGG_SHEET_MAP   = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = '../output/barebones_ragu_v2_individual.xlsx'

METRIC_ROWS = [
    ('Model Score',                    'model_score'),
    ('Gross Loss Impact',              'gross_loss_impact'),
    ('Recovery Impact',                'recovery_impact'),
    ('LTV Impact',                     'ltv_impact'),
    ('APR Impact',                     'apr_impact'),
    ('RAGU Score',                     'ragu_score'),
    ('Amount Financed',                'amt_financed'),
    ('Weighted LTV',                   'ltv'),
    ('Weighted APR',                   'apr'),
]

# --- Individual account-level sheet ---
indiv_sheet = EXCEL_SHEET_MAP[granularity]

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if indiv_sheet in wb.sheetnames:
        del wb[indiv_sheet]
    ws = wb.create_sheet(indiv_sheet, 0)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = indiv_sheet

header = ['account_number', 'lob', 'vintage', 'book_date', 'app_date',
          'model_score', 'unit_loss', 'gross_loss_impact', 'recovery_impact',
          'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed',
          'ltv', 'apr', 'recovery_multiplier', 'loss_multiplier', 'con_term']
for col_idx, col_name in enumerate(header, start=1):
    ws.cell(row=1, column=col_idx, value=col_name)

for row_idx, row in enumerate(acct_df.itertuples(index=False), start=2):
    for col_idx, val in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=val)

print(f"Individual sheet '{indiv_sheet}': {len(acct_df):,} rows")

# --- Aggregated vintage-LOB sheet (same pivot format as V2) ---
agg_sheet = AGG_SHEET_MAP[granularity]
if agg_sheet in wb.sheetnames:
    del wb[agg_sheet]
ws_agg = wb.create_sheet(agg_sheet)

sorted_vintages = sorted(vintage_lob_df['vintage'].unique())
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())
current_row = 1

for lob in all_export_lobs:
    lob_data = vintage_lob_df[vintage_lob_df.lob == lob].set_index('vintage')

    ws_agg.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws_agg.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws_agg.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws_agg.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Aggregated sheet '{agg_sheet}': {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")
print(f"Saved to {EXCEL_OUTPUT}")

# --- Redshift Upload ---

run_sandbox = True
run_historical = True

col_rename = {
    'model_score': 'model_score',
    'gross_loss_impact': 'gross_loss',
    'recovery_impact': 'recovery',
    'ltv_impact': 'ltv',
    'apr_impact': 'apr',
}

output_df = vintage_lob_df.copy()
output_df['month_run'] = pd.Timestamp.now().strftime('%Y M%m')
output_df = output_df.rename(columns=col_rename)


def upload_ragu_to_redshift(df, table='sandbox.ragu_v2_individual_monthend_current'):
    """Upload a DataFrame to a Redshift table via INSERT INTO VALUES."""
    upload_df = df.copy().reset_index(drop=True)
    upload_df['insert_column'] = (
        "('" + upload_df['lob'].astype(str)
        + "', '" + upload_df['vintage'].astype(str)
        + "', " + upload_df['model_score'].round(4).astype(str)
        + ", " + upload_df['gross_loss'].round(4).astype(str)
        + ", " + upload_df['recovery'].round(4).astype(str)
        + ", " + upload_df['ltv'].round(4).astype(str)
        + ", " + upload_df['apr'].round(4).astype(str)
        + ", " + upload_df['ragu_score'].round(4).astype(str)
        + ", '" + upload_df['month_run'].astype(str)
        + "')"
    )
    values_str = upload_df['insert_column'].str.cat(sep=',').replace("'nan'", 'null')

    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {table};")
        cur.execute(f"""
            CREATE TABLE {table} (
                lob         VARCHAR(25),
                vintage     VARCHAR(20),
                model_score FLOAT,
                gross_loss  FLOAT,
                recovery    FLOAT,
                ltv         FLOAT,
                apr         FLOAT,
                ragu_score  FLOAT,
                month_run   VARCHAR(10)
            );
        """)
        cur.execute(f"INSERT INTO {table} VALUES {values_str}")
        conn.commit()
    print(f"Uploaded {len(upload_df)} rows to {table}")


def upload_ragu_historical(month_run_val, table='sandbox.ragu_v2_individual_monthend',
                           source='sandbox.ragu_v2_individual_monthend_current'):
    """Append current-table rows into historical with current_version_flag."""
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {table} (
                lob                  VARCHAR(25),
                vintage              VARCHAR(20),
                model_score          FLOAT,
                gross_loss           FLOAT,
                recovery             FLOAT,
                ltv                  FLOAT,
                apr                  FLOAT,
                ragu_score           FLOAT,
                month_run            VARCHAR(10),
                current_version_flag SMALLINT DEFAULT 0
            );
        """)

        cur.execute(f"DELETE FROM {table} WHERE month_run = '{month_run_val}'")
        cur.execute(f"UPDATE {table} SET current_version_flag = 0 WHERE current_version_flag = 1")
        cur.execute(f"INSERT INTO {table} SELECT *, 1 FROM {source}")
        conn.commit()
    print(f"Historical table {table} updated (month_run={month_run_val}, flag=1)")

if run_sandbox:
    if granularity != 'm':
        print(f"Sandbox upload skipped (granularity='{granularity}'). Only runs for monthly ('m').")
    else:
        month_run_val = output_df['month_run'].iloc[0]
        sandbox_df = output_df[output_df['vintage'] < month_run_val].drop(columns='amt_financed')
        print(f"Filtered to {len(sandbox_df)} rows (excluded vintages >= {month_run_val})")
        upload_ragu_to_redshift(sandbox_df)

if run_historical:
    upload_ragu_historical(output_df['month_run'].iloc[0])

print("[PROGRESS] Export Complete")

Individual sheet 'Individual (Q)': 863,759 rows
Aggregated sheet 'Data Tables (Q)': 9 groups x 27 periods
Saved to barebones_ragu_v2_individual.xlsx
Sandbox upload skipped (granularity='q'). Only runs for monthly ('m').


ProgrammingError: ('42P01', '[42P01] [Redshift][ODBC Driver][Server]42P01:ERROR:  relation "sandbox.ragu_v2_individual_monthend_current" does not exist\n (0) (SQLExecDirectW)')

In [ ]:
# =============================================================================
# CELL 10: AGGREGATE CSV EXPORT + ACCOUNT-LEVEL REDSHIFT UPLOAD
# =============================================================================

# --- Aggregated scores CSV ---
agg_csv = vintage_lob_df.copy()
agg_csv['month_run'] = pd.Timestamp.now().strftime('%Y M%m')
agg_csv.to_csv('all_df_v2_individual.csv', index=False)
print(f"Saved all_df_v2_individual.csv ({len(agg_csv)} rows)")

display(agg_csv.sort_values(['lob', 'vintage']).head(20))

# --- Account-level Redshift upload to sandbox.ragu_v2_ind ---

run_ind_upload = True

def upload_acct_to_redshift(df, table='sandbox.ragu_v2_ind', chunk_size=5000):
    """Upload account-level DataFrame to Redshift via batched INSERT INTO VALUES."""
    upload_df = df[['account_number', 'lob', 'vintage', 'book_date', 'app_date',
                     'model_score', 'unit_loss', 'gross_loss_impact', 'recovery_impact',
                     'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed',
                     'ltv', 'apr', 'recovery_multiplier', 'loss_multiplier', 'con_term']].copy()
    upload_df['month_run'] = pd.Timestamp.now().strftime('%Y M%m')

    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {table};")
        cur.execute(f"""
            CREATE TABLE {table} (
                account_number      VARCHAR(30),
                lob                 VARCHAR(25),
                vintage             VARCHAR(20),
                book_date           DATE,
                app_date            DATE,
                model_score         FLOAT,
                unit_loss           FLOAT,
                gross_loss_impact   FLOAT,
                recovery_impact     FLOAT,
                ltv_impact          FLOAT,
                apr_impact          FLOAT,
                ragu_score          FLOAT,
                amt_financed        FLOAT,
                ltv                 FLOAT,
                apr                 FLOAT,
                recovery_multiplier FLOAT,
                loss_multiplier     FLOAT,
                con_term            FLOAT,
                month_run           VARCHAR(10)
            );
        """)

        for start in range(0, len(upload_df), chunk_size):
            chunk = upload_df.iloc[start:start + chunk_size]
            rows = []
            for _, r in chunk.iterrows():
                vals = []
                for col in upload_df.columns:
                    v = r[col]
                    if pd.isna(v):
                        vals.append('null')
                    elif isinstance(v, str):
                        vals.append(f"'{v}'")
                    elif isinstance(v, (pd.Timestamp, dt.date)):
                        vals.append(f"'{v}'")
                    else:
                        vals.append(str(round(v, 6)))
                rows.append(f"({','.join(vals)})")
            values_str = ','.join(rows)
            cur.execute(f"INSERT INTO {table} VALUES {values_str}")

        conn.commit()
    print(f"Uploaded {len(upload_df):,} rows to {table}")

if run_ind_upload:
    if granularity != 'm':
        print(f"Account upload skipped (granularity='{granularity}'). Only runs for monthly ('m').")
    else:
        upload_acct_to_redshift(acct_df)

print("[PROGRESS] Cell 10 Complete")